# Evaluation

### In this notebook, we evaluate BestModel performance combared to our baseline model

### Imports and Data Loading

In [6]:
import os
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from urllib import request
from pathlib import Path

In [19]:
module_url = f"https://raw.githubusercontent.com/Perez-AlmendrosC/dontpatronizeme/master/semeval-2022/dont_patronize_me.py"
module_name = module_url.split('/')[-1]
print(f'Fetching {module_url}')
#with open("file_1.txt") as f1, open("file_2.txt") as f2
with request.urlopen(module_url) as f, open(module_name,'w') as outf:
  a = f.read()
  outf.write(a.decode('utf-8'))

from dont_patronize_me import DontPatronizeMe

dpm = DontPatronizeMe('.', "./task4_test.tsv")

dpm.load_task1()

train_ids = pd.read_csv('train_semeval_parids-labels.csv')
test_ids = pd.read_csv('dev_semeval_parids-labels.csv')

train_ids.par_id = train_ids.par_id.astype(str)
test_ids.par_id = test_ids.par_id.astype(str)

data=dpm.train_task1_df

Fetching https://raw.githubusercontent.com/Perez-AlmendrosC/dontpatronizeme/master/semeval-2022/dont_patronize_me.py


In [20]:
data_merged = data.copy()
data_merged["par_id"] = data_merged["par_id"].astype(str)

dev_df = test_ids.merge(data_merged, on="par_id", how="left", suffixes=("_csv", "_binary"))

keep_cols = ["par_id", "text", "label_binary", "label_csv", "orig_label"]
dev_df = dev_df[keep_cols].copy()


In [21]:
def load_preds_txt(path: str | Path, expected_len: int | None = None, name: str = "preds"):
    path = Path(path)
    lines = path.read_text(encoding="utf-8").splitlines()
    lines = [ln.strip() for ln in lines if ln.strip() != ""]  # drop blank lines

    preds = [int(ln) for ln in lines]

    return preds

In [22]:
n = len(dev_df)
dev_df["pred_baseline"] = load_preds_txt("exp5.txt", expected_len=n, name="baseline")
dev_df["pred_best"]     = load_preds_txt("dev.txt", expected_len=n, name="bestmodel")

In [26]:
import ast

SUBTYPES = [
    "Unbalanced power relations",
    "Shallow solution",
    "Presupposition",
    "Authority voice",
    "Metaphor",
    "Compassion",
    "The poorer, the merrier",
]

# Ensure label_csv is a list 
if isinstance(dev_df.loc[0, "label_csv"], str):
    dev_df["label_csv"] = dev_df["label_csv"].apply(ast.literal_eval)

dev_df["subtypes_active"] = dev_df["label_csv"].apply(
    lambda v: [SUBTYPES[i] for i, x in enumerate(v) if x == 1]
)

# Ensure types are ints
dev_df["label_binary"] = dev_df["label_binary"].astype(int)
dev_df["pred_baseline"] = dev_df["pred_baseline"].astype(int)
dev_df["pred_best"] = dev_df["pred_best"].astype(int)

In [27]:
dev_df

,par_id,text,label_binary,label_csv,orig_label,pred_baseline,pred_best,subtypes_active
0,4046,We also know that they can benefit by receivin...,1,"[1, 0, 0, 1, 0, 0, 0]",3,0,0,"[Unbalanced power relations, Authority voice]"
1,1279,Pope Francis washed and kissed the feet of Mus...,1,"[0, 1, 0, 0, 0, 0, 0]",4,1,1,[Shallow solution]
2,8330,Many refugees do n't want to be resettled anyw...,1,"[0, 0, 1, 0, 0, 0, 0]",2,0,0,[Presupposition]
3,4063,"""Budding chefs , like """" Fred """" , """" Winston ...",1,"[1, 0, 0, 1, 1, 1, 0]",4,1,1,"[Unbalanced power relations, Authority voice, ..."
4,4089,"""In a 90-degree view of his constituency , one...",1,"[1, 0, 0, 0, 0, 0, 0]",3,0,0,[Unbalanced power relations]
...,...,...,...,...,...,...,...,...
2089,10462,"The sad spectacle , which occurred on Saturday...",0,"[0, 0, 0, 0, 0, 0, 0]",0,0,0,[]
2090,10463,""""""" The Pakistani police came to our house and...",0,"[0, 0, 0, 0, 0, 0, 0]",0,0,0,[]
2091,10464,"""When Marie O'Donoghue went looking for a spec...",0,"[0, 0, 0, 0, 0, 0, 0]",0,0,0,[]
2092,10465,"""Sri Lankan norms and culture inhibit women fr...",0,"[0, 0, 0, 0, 0, 0, 0]",1,0,0,[]


### Below we compute different evaluation metrics

In [32]:
from sklearn.metrics import precision_recall_fscore_support, accuracy_score, confusion_matrix

def model_metrics(y_true, y_pred):
    p, r, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", pos_label=1, zero_division=0)
    acc = accuracy_score(y_true, y_pred)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0,1]).ravel()
    return {
        "TP": tp, "FP": fp, "FN": fn, "TN": tn,
        "Accuracy": acc,
        "Precision": p,
        "Recall": r,
        "F1": f1 
    }

m_base = model_metrics(dev_df["label_binary"], dev_df["pred_baseline"])
m_best = model_metrics(dev_df["label_binary"], dev_df["pred_best"])

metrics_df = pd.DataFrame([m_base, m_best], index=["Baseline", "BestModel"])
metrics_df.round(4)

,TP,FP,FN,TN,Accuracy,Precision,Recall,F1
Baseline,112,89,87,1806,0.9160,0.5572,0.5628,0.5600
BestModel,105,59,94,1836,0.9269,0.6402,0.5276,0.5785


### Now we examine improvment from. baseline to BestModel

In [34]:
def delta_group(row):
    gold = row["label_binary"]
    b = row["pred_baseline"]
    m = row["pred_best"]
    base_correct = (b == gold)
    best_correct = (m == gold)

    if (not base_correct) and best_correct:
        return "Improved in BestModel"
    if base_correct and (not best_correct):
        return "Regression from Baseline"
    if (not base_correct) and (not best_correct):
        return "Both wrong"
    return "Both correct"

dev_df["delta_group"] = dev_df.apply(delta_group, axis=1)

delta_counts = dev_df["delta_group"].value_counts().reindex(
    ["Improved in BestModel", "Regression from Baseline", "Both wrong", "Both correct"],
    fill_value=0
)
delta_counts

delta_group
Improved in BestModel         47
Regression from Baseline      24
Both wrong                   129
Both correct                1894
Name: count, dtype: int64

### Below we evalue the model correctness for each PCL catagory

In [39]:
def fn_rate_for_subtype(df, subtype_idx, pred_col):
    # Gold positives for this subtype
    mask = df["label_csv"].apply(lambda v: v[subtype_idx] == 1)
    sub = df[mask]
    if len(sub) == 0:
        return np.nan, 0
    # FN among those (gold is positive by construction; FN means predicted 0)
    fn = (sub[pred_col] == 0).sum()
    return fn / len(sub), len(sub)

rows = []
for i, name in enumerate(SUBTYPES):
    base_rate, support = fn_rate_for_subtype(dev_df, i, "pred_baseline")
    best_rate, _       = fn_rate_for_subtype(dev_df, i, "pred_best")
    rows.append({
        "Subtype": name,
        "# of examples in dev set": support,
        "Baseline FN rate": base_rate,
        "BestModel FN rate": best_rate,
        "Δ FN rate (Best - Base)": (best_rate - base_rate) if (support > 0) else np.nan,
    })

subtype_df = pd.DataFrame(rows).sort_values(
    by=["BestModel FN rate", "# of examples in dev set"],
    ascending=[False, False]
)

subtype_df

,Subtype,# of examples in dev set,Baseline FN rate,BestModel FN rate,Δ FN rate (Best - Base)
6,"The poorer, the merrier",11,0.636364,0.636364,0.000000
2,Presupposition,62,0.500000,0.580645,0.080645
3,Authority voice,38,0.447368,0.447368,0.000000
5,Compassion,106,0.377358,0.433962,0.056604
4,Metaphor,52,0.365385,0.423077,0.057692
0,Unbalanced power relations,142,0.366197,0.401408,0.035211
1,Shallow solution,36,0.388889,0.333333,-0.055556


### Next we evaluate correctness compared to the original labels (0-4)

In [41]:
def by_severity_table(df):
    out = []
    for s in sorted(df["orig_label"].unique()):
        sub = df[df["orig_label"] == s]
        n = len(sub)
        if n == 0:
            continue
        gold_pos = sub["label_binary"].mean()
        base_pos = sub["pred_baseline"].mean()
        best_pos = sub["pred_best"].mean()
        out.append({
            "original label": int(s),
            "Count": n,
            "True PCL rate": gold_pos,
            "Baseline PCL rate": base_pos,
            "BestModel PCL rate": best_pos,
        })
    return pd.DataFrame(out)

sev_df = by_severity_table(dev_df)
sev_df

,original label,Count,True PCL rate,Baseline PCL rate,BestModel PCL rate
0,0,1704,0.0,0.028756,0.018779
1,1,191,0.0,0.209424,0.141361
2,2,18,1.0,0.222222,0.222222
3,3,89,1.0,0.494382,0.404494
4,4,92,1.0,0.695652,0.706522


### Below we extract different paragraphs where models failed

In [42]:
fixed_mask = (dev_df["pred_baseline"] != dev_df["label_binary"]) & (dev_df["pred_best"] == dev_df["label_binary"])
reg_mask   = (dev_df["pred_baseline"] == dev_df["label_binary"]) & (dev_df["pred_best"] != dev_df["label_binary"])
both_wrong = (dev_df["pred_baseline"] != dev_df["label_binary"]) & (dev_df["pred_best"] != dev_df["label_binary"])

print("Fixed by BestModel:", fixed_mask.sum())
print("Regressions:", reg_mask.sum())
print("Both wrong:", both_wrong.sum())

Fixed by BestModel: 47
Regressions: 24
Both wrong: 129


In [43]:
import textwrap

def show_examples(df, mask, k=5, title="Examples"):
    sub = df.loc[mask, ["par_id", "orig_label", "label_binary", "label_csv", "subtypes_active", "pred_baseline", "pred_best", "text"]].copy()
    # shorter excerpt for readability
    sub["excerpt"] = sub["text"].apply(lambda t: textwrap.shorten(str(t), width=220, placeholder="..."))
    # show most informative first: positives (label 3/4) and multi-subtype cases
    sub["num_subtypes"] = sub["label_csv"].apply(sum)
    sub = sub.sort_values(by=["label_binary", "orig_label", "num_subtypes"], ascending=[False, False, False])
    display(sub.head(k)[["par_id","orig_label","label_binary","pred_baseline","pred_best","subtypes_active","excerpt"]])
    return sub

fixed_df = show_examples(dev_df, fixed_mask, k=6, title="Fixed by BestModel")
reg_df   = show_examples(dev_df, reg_mask,   k=4, title="Regressions")
both_df  = show_examples(dev_df, both_wrong, k=4, title="Both wrong")

,par_id,orig_label,label_binary,pred_baseline,pred_best,subtypes_active,excerpt
90,5612,4,1,0,1,"[Unbalanced power relations, Shallow solution,...",He said the victims who are currently rendered...
151,3861,4,1,0,1,"[Unbalanced power relations, Metaphor, Compass...","""In time , when the housing backlog for the lo..."
87,6907,4,1,0,1,"[Unbalanced power relations, Presupposition]",""""""" We are taking a community that is fairly v..."
135,1200,4,1,0,1,"[Unbalanced power relations, Shallow solution]","On the eve of the World Refugee Day , UNHCR re..."
160,10269,4,1,0,1,"[Metaphor, Compassion]",Veterans left on scrapheap : The homeless plig...
176,2453,4,1,0,1,"[Authority voice, Compassion]",""""""" My daughter , who was a physiotherapist , ..."


,par_id,orig_label,label_binary,pred_baseline,pred_best,subtypes_active,excerpt
132,4580,4,1,1,0,"[Unbalanced power relations, Presupposition, A...","Old BC : As usual , Old GB has it assy-versy ...."
46,6821,4,1,1,0,"[Unbalanced power relations, Presupposition, C...","That is the big picture , the one conveyed in ..."
61,10007,4,1,1,0,"[Unbalanced power relations, Presupposition, C...",It is seen in recurring violence and continuin...
77,10253,4,1,1,0,"[Presupposition, Metaphor, Compassion]",Rather sad . Good set of pictures that tells a...


,par_id,orig_label,label_binary,pred_baseline,pred_best,subtypes_active,excerpt
104,157,4,1,0,0,"[Unbalanced power relations, Shallow solution,...",We are alarmed to learn of your recently circu...
178,5955,4,1,0,0,"[Presupposition, Authority voice, Metaphor, Co...",The indigenous Palestinians see themselves aba...
13,1572,4,1,0,0,"[Unbalanced power relations, Compassion, The p...",If only we had more stories that championed th...
40,3661,4,1,0,0,"[Unbalanced power relations, Presupposition, A...",""""""" Some come from poor families and then you ..."


In [44]:
# False positives: gold 0, predicted 1
base_fp = (dev_df["label_binary"] == 0) & (dev_df["pred_baseline"] == 1)
best_fp = (dev_df["label_binary"] == 0) & (dev_df["pred_best"] == 1)

# False negatives: gold 1, predicted 0
base_fn = (dev_df["label_binary"] == 1) & (dev_df["pred_baseline"] == 0)
best_fn = (dev_df["label_binary"] == 1) & (dev_df["pred_best"] == 0)

print("Baseline FP:", base_fp.sum(), " BestModel FP:", best_fp.sum())
print("Baseline FN:", base_fn.sum(), " BestModel FN:", best_fn.sum())

# Show a few of each (especially useful for writing)
_ = show_examples(dev_df, best_fp, k=3, title="BestModel false positives")
_ = show_examples(dev_df, best_fn, k=3, title="BestModel false negatives")

Baseline FP: 89  BestModel FP: 59
Baseline FN: 87  BestModel FN: 94


,par_id,orig_label,label_binary,pred_baseline,pred_best,subtypes_active,excerpt
413,8616,1,0,0,1,[],""""""" We want all poor families to earn enough a..."
421,8626,1,0,1,1,[],""""""" Your personal leadership has been critical..."
614,8838,1,0,1,1,[],""""""" I always consider this job as a gift , bei..."


,par_id,orig_label,label_binary,pred_baseline,pred_best,subtypes_active,excerpt
104,157,4,1,0,0,"[Unbalanced power relations, Shallow solution,...",We are alarmed to learn of your recently circu...
132,4580,4,1,1,0,"[Unbalanced power relations, Presupposition, A...","Old BC : As usual , Old GB has it assy-versy ...."
178,5955,4,1,0,0,"[Presupposition, Authority voice, Metaphor, Co...",The indigenous Palestinians see themselves aba...
